# Deep Q-Networks

**Companion lesson:** https://ml-viz.vercel.app/courses/reinforcement-learning/03-deep-q-networks

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## DQN machinery in numpy

When states are too many to tabulate, approximate $Q$ with a network. We build a 2-layer MLP Q-network (one-hot state in, 4 Q-values out) plus the two tricks that make DQN stable: an **experience replay** buffer and a **target network**.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)

def one_hot(s, n):
    v = np.zeros(n); v[s] = 1; return v

def relu(x): return np.maximum(0, x)

class QNet:
    def __init__(self, n_in, nh, n_out, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(n_in, nh)*np.sqrt(2/n_in); self.b1 = np.zeros(nh)
        self.W2 = rng.randn(nh, n_out)*np.sqrt(2/nh);  self.b2 = np.zeros(n_out)
    def forward(self, X):
        self.X = X; self.z1 = X@self.W1 + self.b1; self.h = relu(self.z1)
        return self.h @ self.W2 + self.b2
    def copy_from(self, other):
        self.W1, self.b1 = other.W1.copy(), other.b1.copy()
        self.W2, self.b2 = other.W2.copy(), other.b2.copy()
print('Q-network ready')

## Training loop: replay + target net

Each step is stored in the buffer; we train on random minibatches whose targets come from a **frozen** target network, synced every few hundred steps.

In [ ]:
from collections import deque
import random

def train_dqn(env, episodes=600, gamma=0.9, lr=0.01, nh=64, batch=32, sync=200):
    rng = np.random.RandomState(0); random.seed(0)
    q, qt = QNet(env.nS, nh, env.nA), QNet(env.nS, nh, env.nA)
    qt.copy_from(q)
    buf = deque(maxlen=5000); eps, step, returns = 1.0, 0, []
    for ep in range(episodes):
        s, done, total, t = 0, False, 0, 0
        while not done and t < 100:
            a = rng.randint(env.nA) if rng.rand()<eps else int(np.argmax(q.forward(one_hot(s,env.nS))))
            s2, r, done = env.step(s, a)
            buf.append((s,a,r,s2,done)); s=s2; total+=r; t+=1; step+=1
            if len(buf) >= batch:
                B = random.sample(buf, batch)
                S  = np.array([one_hot(b[0],env.nS) for b in B])
                S2 = np.array([one_hot(b[3],env.nS) for b in B])
                A = np.array([b[1] for b in B]); R = np.array([b[2] for b in B])
                D = np.array([b[4] for b in B], float)
                target = R + gamma*qt.forward(S2).max(1)*(1-D)   # frozen target net
                pred = q.forward(S)                              # caches activations
                dout = np.zeros_like(pred)
                dout[np.arange(batch), A] = (pred[np.arange(batch),A] - target)/batch
                # backprop through the 2-layer MLP
                dW2 = q.h.T@dout; db2 = dout.sum(0)
                dh = dout@q.W2.T; dz1 = dh*(q.z1>0)
                dW1 = S.T@dz1; db1 = dz1.sum(0)
                q.W2-=lr*dW2; q.b2-=lr*db2; q.W1-=lr*dW1; q.b1-=lr*db1
            if step % sync == 0: qt.copy_from(q)
        eps = max(0.05, eps*0.99); returns.append(total)
    return q, returns

q, returns = train_dqn(env)
print('trained DQN over', len(returns), 'episodes')

## Did it learn? Greedy rollout

In [ ]:
ma = np.convolve(returns, np.ones(30)/30, mode='valid')
plt.plot(ma, color='#6366f1'); plt.xlabel('episode'); plt.ylabel('return (30-ep avg)')
plt.title('DQN learning curve on the gridworld'); plt.show()

s, path, done = 0, [0], False
while not done and len(path) < 25:
    s, _, done = env.step(s, int(np.argmax(q.forward(one_hot(s,env.nS))))); path.append(s)
print('greedy path:', path)
print('reached goal:', path[-1]==env.goal, 'in', len(path)-1, 'steps')

## Key takeaways

- A **Q-network** replaces the table so values generalize across states.
- **Experience replay** decorrelates the data fed to gradient descent.
- A **target network** keeps the regression target from chasing its own tail.
- Remove either trick and the same loop becomes unstable.

## Experience Replay Demo

In [ ]:
import random
from collections import deque

class ReplayBuffer:
    """Minimal replay buffer storing (s, a, r, s_next, done) tuples."""
    def __init__(self, maxlen=10_000):
        self._buf = deque(maxlen=maxlen)

    def push(self, transition):
        """Store a single (s, a, r, s_next, done) transition."""
        self._buf.append(transition)

    def sample(self, batch_size):
        """Return a list of batch_size randomly sampled transitions."""
        return random.sample(self._buf, batch_size)

    def __len__(self):
        return len(self._buf)


rng = random.Random(42)
buf = ReplayBuffer(maxlen=10_000)

# Generate 100 random transitions: states 0-3, actions 0-1, reward -1 or +1
for _ in range(100):
    s      = rng.randint(0, 3)
    a      = rng.randint(0, 1)
    r      = rng.choice([-1, 1])
    s_next = rng.randint(0, 3)
    done   = False
    buf.push((s, a, r, s_next, done))

batch = buf.sample(16)
print(f"Buffer size: {len(buf)}  |  Batch size sampled: {len(batch)}")
print("
First 3 transitions (s, a, r, s_next, done):")
for t in batch[:3]:
    print(" ", t)

## Why Replay Breaks Correlations

In [ ]:
import numpy as np

# --- Sequential rollout: state increments each step, reward = state mod 2 ---
seq_rewards = [s % 2 for s in range(10)]

def consecutive_correlation(seq):
    """Pearson correlation between seq[:-1] and seq[1:] (consecutive pairs)."""
    x = np.array(seq[:-1], dtype=float)
    y = np.array(seq[1:],  dtype=float)
    if np.std(x) == 0 or np.std(y) == 0:
        return float('nan')
    return float(np.corrcoef(x, y)[0, 1])

seq_corr = consecutive_correlation(seq_rewards)
print(f"Sequential rollout rewards : {seq_rewards}")
print(f"Consecutive correlation    : {seq_corr:.3f}  (high — ~1.0 or -1.0)")

# --- Replay buffer: same 10 transitions stored, but sampled randomly ---
buf2 = ReplayBuffer(maxlen=10_000)
rng2 = random.Random(7)
for step_i in range(10):
    buf2.push((step_i, 0, step_i % 2, (step_i + 1) % 10, False))

sampled = buf2.sample(10)
buf_rewards = [t[2] for t in sampled]  # rewards in random order
buf_corr = consecutive_correlation(buf_rewards)
print(f"
Buffer-sampled rewards     : {buf_rewards}")
print(f"Consecutive correlation    : {buf_corr:.3f}  (near 0 — decorrelated)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Build a large pool of transitions with alternating reward pattern
pool_size = 200
all_rewards = np.array([i % 2 for i in range(pool_size)], dtype=float)

fill_fracs  = np.linspace(0.05, 1.0, 40)   # buffer fill level: 5% to 100%
chunk_size  = 10                             # compare chunks of this size

seq_corrs = []
buf_corrs = []

rng3 = np.random.RandomState(0)

for frac in fill_fracs:
    n_avail = max(chunk_size + 1, int(frac * pool_size))
    pool_slice = all_rewards[:n_avail]

    # Sequential: last chunk_size rewards in order
    seq_chunk = pool_slice[-chunk_size:].tolist()
    seq_c = consecutive_correlation(seq_chunk)
    seq_corrs.append(abs(seq_c) if not np.isnan(seq_c) else 0.0)

    # Buffer: random sample of chunk_size from the available pool
    idx = rng3.choice(n_avail, size=chunk_size, replace=False)
    buf_chunk = pool_slice[idx].tolist()
    buf_c = consecutive_correlation(buf_chunk)
    buf_corrs.append(abs(buf_c) if not np.isnan(buf_c) else 0.0)

fig, ax = plt.subplots()
ax.plot([f * 100 for f in fill_fracs], seq_corrs,
        color='#f97316', linewidth=2, label='Sequential (high correlation)')
ax.plot([f * 100 for f in fill_fracs], buf_corrs,
        color='#6366f1', linewidth=2, label='Replay buffer (near-zero)')
ax.set_xlabel('Replay buffer fill level (%)')
ax.set_ylabel('|Consecutive correlation|')
ax.set_title('Experience Replay Breaks Temporal Correlations')
ax.legend(facecolor='#1a1d27', edgecolor='#334155', labelcolor='#e2e8f0')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1 — DQN target computation

The DQN loss target for a transition $(s, a, r, s', \\text{done})$ is:

$$y = r + \\gamma \\, \\max_{a'} Q_{\\text{target}}(s', a') \\cdot (1 - \\text{done})$$

The $(1-\\text{done})$ term zeros out the future-reward term for terminal transitions.
Implement the vectorised version and verify on the three-element example.

In [ ]:
import numpy as np

def compute_targets(rewards, q_next_max, dones, gamma):
    """Vectorised DQN target computation.
    rewards: (N,) array of rewards.
    q_next_max: (N,) array of max Q-values from the target network at s'.
    dones: (N,) boolean array indicating terminal transitions.
    gamma: float discount factor.
    Returns: (N,) array of targets y."""
    # TODO(you): implement the formula, zeroing out future reward at terminal steps
    ...

In [ ]:
r   = np.array([-1.0, 10.0, -1.0])
qn  = np.array([2.0,   0.0,  3.0])
d   = np.array([False, True, False])
y   = compute_targets(r, qn, d, gamma=0.9)

assert y.shape == (3,), "must return array of shape (N,)"
assert abs(y[0] - 0.8)  < 1e-9, "non-terminal: -1 + 0.9*2 = 0.8"
assert abs(y[1] - 10.0) < 1e-9, "terminal: no future reward, y = r = 10"
assert abs(y[2] - 1.7)  < 1e-9, "non-terminal: -1 + 0.9*3 = 1.7"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compute_targets(rewards, q_next_max, dones, gamma):
    return rewards + gamma * q_next_max * (~dones).astype(float)
```

</details>

### Exercise 2 — Experience replay basics

The replay buffer is a fixed-size deque: when full, the oldest transition is dropped.
Random minibatch sampling breaks the temporal correlation that would bias gradient
descent. Implement a minimal buffer and verify the capacity cap and sample size.

In [ ]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, maxlen):
        # TODO(you): create a deque with the given maxlen
        ...

    def push(self, transition):
        """Add a transition (s, a, r, s', done) to the buffer."""
        # TODO(you): append to the deque
        ...

    def sample(self, batch_size):
        """Return a list of batch_size randomly sampled transitions."""
        # TODO(you): use random.sample
        ...

    def __len__(self):
        return len(self.buffer)

In [ ]:
random.seed(0)
buf = ReplayBuffer(maxlen=10)

for i in range(15):
    buf.push((i, 0, -1.0, i + 1, False))

assert len(buf) == 10, \
    "buffer must cap at maxlen=10 even after 15 pushes"
sample = buf.sample(5)
assert len(sample) == 5, \
    "sample(5) must return exactly 5 transitions"
# oldest 5 transitions (indices 0-4) should be evicted
stored_states = {t[0] for t in buf.sample(10)}
assert all(s >= 5 for s in stored_states), \
    "transitions with index < 5 should be evicted (buffer overflow)"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
class ReplayBuffer:
    def __init__(self, maxlen):
        self.buffer = deque(maxlen=maxlen)

    def push(self, transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)
```

</details>